# Top 100 Pre-1900 Face-Confidence Pages

This notebook selects the 100 pre-1900 Economist archive pages with the highest face-detection segmentation confidence in the deduplicated face dataset.

Run this notebook from its own directory, `code/scripts`. The repository's VS Code setting `jupyter.notebookFileRoot = ${fileDirname}` makes that the default in VS Code/Jupyter.

Input:

`../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv`

Output:

`../../data/processed/top_100_pre_1900_face_confidence_pages.json`

The output is a plain JSON list of page identifiers in `yyyy-mmdd-pppp` format.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd


pd.options.display.max_rows = 120
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 160

# Paths are relative to this notebook's directory: code/scripts.
deduplicated_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
output_json = Path("../../data/processed/top_100_pre_1900_face_confidence_pages.json")

expected_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

page_id_pattern = re.compile(r"^\d{4}-\d{4}-\d{4}$")


## Load and Validate Data

The input is the deduplicated face-detection table. The notebook keeps the existing row schema and adds only temporary derived fields needed to assign detections to page IDs.

In [ ]:
faces = pd.read_csv(deduplicated_csv, dtype={"Filename": "string"})

assert list(faces.columns) == expected_columns, {
    "expected": expected_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The deduplicated face CSV is empty."

numeric_columns = [
    "Bounding Box relative X1",
    "Bounding Box relative X2",
    "Segmentation confidence score",
]
for column in numeric_columns:
    faces[column] = pd.to_numeric(faces[column], errors="coerce")

parsed = faces["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)
faces["issue_id"] = parsed["issue_id"]
faces["source_pages"] = parsed["source_pages"]
faces["issue_year"] = faces["issue_id"].str.slice(0, 4).astype("Int64")

parse_failures = int(faces["issue_id"].isna().sum())
assert parse_failures == 0, f"Could not parse {parse_failures:,} filenames."
assert faces["Segmentation confidence score"].notna().all(), "Confidence scores must be numeric."

print(f"Loaded {len(faces):,} deduplicated face detections.")
print(f"Issues covered: {faces['issue_id'].nunique():,}")


## Assign Page IDs

Single-page source scans use their encoded page directly. For multi-page scans, the detection center along the x-axis assigns the face to the left-to-right page segment, matching the convention used in the archive analysis notebooks.

In [ ]:
faces["center_x"] = (
    faces["Bounding Box relative X1"] + faces["Bounding Box relative X2"]
) / 2


def assigned_page_number(row: pd.Series) -> int:
    pages = [int(page) for page in str(row["source_pages"]).split(",")]
    if len(pages) == 1 or pd.isna(row["center_x"]):
        return pages[0]
    page_index = int(max(0, min(0.999999, float(row["center_x"]))) * len(pages))
    return pages[page_index]


faces["assigned_page_number"] = faces.apply(assigned_page_number, axis=1).astype("int64")
faces["page_id"] = faces["issue_id"] + "-" + faces["assigned_page_number"].map(lambda value: f"{value:04d}")

invalid_page_ids = faces.loc[~faces["page_id"].str.match(page_id_pattern), "page_id"]
assert invalid_page_ids.empty, invalid_page_ids.head().tolist()

pd.Series(
    {
        "detections": len(faces),
        "estimated_pages_with_faces": faces["page_id"].nunique(),
        "multi_page_source_scan_detections": int(faces["source_pages"].str.contains(",", regex=False).sum()),
    }
)


## Select Top 100 Pages

Pages are ranked by the highest segmentation confidence among their deduplicated face detections. Ties are resolved by page ID for deterministic output.

In [ ]:
pre_1900_faces = faces.loc[faces["issue_year"] < 1900].copy()
assert len(pre_1900_faces) > 0, "No pre-1900 detections found."

page_confidence = (
    pre_1900_faces.groupby("page_id", as_index=False)
    .agg(max_confidence=("Segmentation confidence score", "max"))
    .sort_values(["max_confidence", "page_id"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

assert len(page_confidence) >= 100, f"Only found {len(page_confidence):,} pre-1900 pages."

top_100_page_confidence = page_confidence.head(100).copy()
top_100_page_ids = top_100_page_confidence["page_id"].tolist()
top_100_min_confidence = top_100_page_confidence["max_confidence"].min()
top_100_max_confidence = top_100_page_confidence["max_confidence"].max()

assert len(top_100_page_ids) == 100
assert len(top_100_page_ids) == len(set(top_100_page_ids)), "Output page IDs must be unique."
assert all(page_id_pattern.fullmatch(page_id) for page_id in top_100_page_ids)

print(f"Selected {len(top_100_page_ids):,} page IDs from {len(page_confidence):,} pre-1900 pages.")
print(
    "Top-100 page-level max confidence range: "
    f"min={top_100_min_confidence:.6f}, max={top_100_max_confidence:.6f}"
)


## Write JSON List

The saved file intentionally contains only the ordered list of page IDs. Confidence values remain reproducible from the input CSV and are not stored in this list artifact.

In [ ]:
output_json.parent.mkdir(parents=True, exist_ok=True)
output_json.write_text(json.dumps(top_100_page_ids, indent=2) + "\n", encoding="utf-8")

reloaded_page_ids = json.loads(output_json.read_text(encoding="utf-8"))
assert reloaded_page_ids == top_100_page_ids

print(f"Wrote {len(top_100_page_ids):,} page IDs to {output_json}")
top_100_page_ids
